# Tests and model evaluation

Two parts: **1. the test suite** (engine, rules and classifier, on synthetic data) and **2. which classification method is best** on your own reviews. Clear the outputs before committing (the pre-commit hook refuses a notebook with outputs).

## 1. Tests

Runs `node --test`, the same suites the pre-commit hook runs. Synthetic data only.

In [ ]:
import re, subprocess
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
run = subprocess.run(["node", "--test"], cwd=REPO, capture_output=True, text=True, encoding="utf-8")
summary = dict(re.findall(r"^# (pass|fail) (\d+)$", run.stdout, re.M))
failed = re.findall(r"^not ok \d+ - (.+)$", run.stdout, re.M)
print(f"passed {summary.get('pass')}, failed {summary.get('fail')}")
for name in failed:
    print("  FAILED:", name)

## 2. Which classification method is best?

Train on the **older 75%** of the transactions you reviewed by hand, test on the **newest 25%**: the methods meet the test rows the way they will meet next month's import.

The methods and the split live in `dashboard/classifier.js`, the same code the dashboard's *Re-run classification* button runs. This notebook only calls `scripts/evaluate.js` and draws the result. It prints aggregate scores only (no descriptions, no amounts), but clear the outputs before committing anyway.

Note: the test set is made of rows **you reviewed**, and you mostly review what the rules could not file. So this measures how well each method handles the hard cases, not how often the rules are right overall.

In [ ]:
import json, subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def as_pct(df):
    """Shares as percentages, for reading."""
    return df.apply(lambda col: col.map(lambda v: "n/a" if pd.isna(v) else f"{v:.0%}"))

REPO = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()

# Set QWEN to a number of test rows to also score the local Qwen (needs Ollama running;
# around 5-15 s per row on a 16 GB Mac). 0 skips it.
QWEN = 0
TEST_SHARE = 0.25

cmd = ["node", "scripts/evaluate.js", "--test-share", str(TEST_SHARE)] + (["--qwen", str(QWEN)] if QWEN else [])
report = json.loads(subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, check=True).stdout)
report["split"]

## Overall

*Accuracy* counts a row right when the method names exactly the category and sub-category you chose. *New merchants* are test rows whose description never appeared in training.

In [ ]:
summary = pd.DataFrame({
    method: {
        "accuracy": r["accuracy"],
        "new merchants": r["newMerchants"]["accuracy"],
        **{f"coverage @{t['threshold']}": t["coverage"] for t in r["thresholds"]},
        **{f"accuracy @{t['threshold']}": t["accuracy"] for t in r["thresholds"]},
    }
    for method, r in report["results"].items()
}).T
as_pct(summary)

In [ ]:
ax = summary[["accuracy", "new merchants"]].plot.bar(figsize=(7, 3.5), rot=0)
ax.set_ylim(0, 1); ax.set_ylabel("share right"); ax.set_title("Accuracy on the newest reviews")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.show()

## Can the confidence be trusted?

For each confidence band, how often the method was right. A trustworthy score climbs with the band: that is what makes "accept everything above 90%" safe.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for method, r in report["results"].items():
    bins = [b for b in r["reliability"] if b["count"]]
    ax.plot([b["from"] + 0.1 for b in bins], [b["accuracy"] for b in bins], marker="o", label=method)
ax.plot([0, 1], [0, 1], color="grey", linestyle=":", linewidth=1, label="perfectly calibrated")
ax.set_xlabel("confidence band (centre)"); ax.set_ylabel("share right"); ax.set_ylim(0, 1.05)
ax.legend(); ax.set_title("Accuracy by confidence")
plt.show()

## Coverage against accuracy

Raising the acceptance threshold files fewer rows automatically but gets more of them right. Pick the threshold for the dashboard's *Accept all above* from this table.

In [ ]:
table = pd.DataFrame([
    {"method": m, "threshold": t["threshold"], "filed automatically": t["coverage"], "of those right": t["accuracy"]}
    for m, r in report["results"].items() for t in r["thresholds"]
])
as_pct(table.pivot(index="threshold", columns="method"))

## By category (best method)

In [ ]:
best = summary["accuracy"].idxmax()
by_cat = pd.DataFrame(report["results"][best]["byCategory"]).T
by_cat["accuracy"] = by_cat["right"] / by_cat["count"]
print(f"best method: {best}")
by_cat["accuracy"] = by_cat["accuracy"].map(lambda v: f"{v:.0%}")
by_cat.sort_values("count", ascending=False)